# CNN 23 - Modelo V3 (Replicado do CNN 1)
Este script foi gerado para executar de forma limpa e sequencial o treinamento exclusivo do Modelo V3.


In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import psutil
import platform
import os

from tensorflow.keras.datasets import cifar100
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv2D,
    MaxPooling2D,
    Dense,
    Dropout,
    BatchNormalization,
    Activation,
    GlobalAveragePooling2D
)
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)
from sklearn.metrics import confusion_matrix, classification_report

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

## 1. Carregamento dos Dados (CIFAR-100)


In [ ]:
(x_train, y_train), (x_test, y_test) = cifar100.load_data()

print("X_train:", x_train.shape)
print("y_train:", y_train.shape)
print("X_test :", x_test.shape)
print("y_test :", y_test.shape)
print("\nNúmero de classes:", len(np.unique(y_train)))

## 2. Pré-processamento e Validação


In [ ]:
# Normalização dos pixels (escala de 0 a 1)
X_train = x_train.astype("float32") / 255.0
X_test = x_test.astype("float32") / 255.0

# One-Hot Encoding para as 100 classes
y_train_cat = to_categorical(y_train, 100)
y_test_cat = to_categorical(y_test, 100)

# Separação de validação (reservando 5000 imagens)
X_val = X_train[:5000]
y_val = y_train_cat[:5000]

X_train_f = X_train[5000:]
y_train_f = y_train_cat[5000:]

print("Treino:", X_train_f.shape, y_train_f.shape)
print("Validação:", X_val.shape, y_val.shape)
print("Teste:", X_test.shape, y_test_cat.shape)

## 3. Data Augmentation


In [ ]:
datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=15,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

datagen.fit(X_train_f)
print("Data Augmentation configurado com sucesso!")

## 4. Arquitetura da Rede (V3)


In [ ]:
model = Sequential()

# ===== BLOCO 1 =====
model.add(Conv2D(64, (3,3), padding='same', input_shape=(32,32,3)))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(64, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

# ===== BLOCO 2 =====
model.add(Conv2D(128, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(128, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

# ===== BLOCO 3 =====
model.add(Conv2D(256, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(256, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.35))

# ===== BLOCO 4 =====
model.add(Conv2D(512, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Conv2D(512, (3,3), padding='same'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.35))

# ===== CLASSIFICADOR =====
model.add(GlobalAveragePooling2D())
model.add(Dense(512))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.5))
model.add(Dense(100, activation='softmax'))

model.summary()

## 5. Compilação e Callbacks (V3)


In [ ]:
# A V3 utiliza learning_rate = 0.001
model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

checkpoint_v3 = ModelCheckpoint(
    'modelo_v3.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print("Modelo compilado e callbacks configurados (Específico V3).")

## 6. Treinamento da V3


In [ ]:
history_v3 = model.fit(
    datagen.flow(
        X_train_f,
        y_train_f,
        batch_size=32
    ),
    # A linha steps_per_epoch foi APAGADA!
    validation_data=(X_val, y_val),
    epochs=150,
    callbacks=[
        early_stop,
        reduce_lr,
        checkpoint_v3
    ],
    verbose=1
)


In [ ]:
Epoch 1/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2221 - loss: 3.1191
Epoch 1: val_accuracy improved from 0.16880 to 0.24820, saving model to modelo_v3.keras

Epoch 1: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.2356 - loss: 3.0461 - val_accuracy: 0.2482 - val_loss: 3.0398 - learning_rate: 0.0010
Epoch 2/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.2758 - loss: 2.8330
Epoch 2: val_accuracy improved from 0.24820 to 0.33160, saving model to modelo_v3.keras

Epoch 2: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.2855 - loss: 2.7879 - val_accuracy: 0.3316 - val_loss: 2.4745 - learning_rate: 0.0010
Epoch 3/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3182 - loss: 2.6117
Epoch 3: val_accuracy did not improve from 0.33160
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.3227 - loss: 2.5897 - val_accuracy: 0.2884 - val_loss: 3.0316 - learning_rate: 0.0010
Epoch 4/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3550 - loss: 2.4560
Epoch 4: val_accuracy improved from 0.33160 to 0.33500, saving model to modelo_v3.keras

Epoch 4: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.3609 - loss: 2.4222 - val_accuracy: 0.3350 - val_loss: 2.5657 - learning_rate: 0.0010
Epoch 5/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3770 - loss: 2.3317
Epoch 5: val_accuracy improved from 0.33500 to 0.36980, saving model to modelo_v3.keras

Epoch 5: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.3841 - loss: 2.3171 - val_accuracy: 0.3698 - val_loss: 2.5625 - learning_rate: 0.0010
Epoch 6/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.4055 - loss: 2.1905
Epoch 6: val_accuracy improved from 0.36980 to 0.40340, saving model to modelo_v3.keras

Epoch 6: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.4073 - loss: 2.1942 - val_accuracy: 0.4034 - val_loss: 2.2472 - learning_rate: 0.0010
Epoch 7/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4298 - loss: 2.0972
Epoch 7: val_accuracy improved from 0.40340 to 0.46800, saving model to modelo_v3.keras

Epoch 7: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.4280 - loss: 2.1067 - val_accuracy: 0.4680 - val_loss: 1.9780 - learning_rate: 0.0010
Epoch 8/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4528 - loss: 2.0255
Epoch 8: val_accuracy did not improve from 0.46800
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.4520 - loss: 2.0269 - val_accuracy: 0.4498 - val_loss: 2.0944 - learning_rate: 0.0010
Epoch 9/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4627 - loss: 1.9599
Epoch 9: val_accuracy did not improve from 0.46800
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.4641 - loss: 1.9611 - val_accuracy: 0.4650 - val_loss: 2.0230 - learning_rate: 0.0010
Epoch 10/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4778 - loss: 1.9134
Epoch 10: val_accuracy improved from 0.46800 to 0.50000, saving model to modelo_v3.keras

Epoch 10: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.4800 - loss: 1.8961 - val_accuracy: 0.5000 - val_loss: 1.7997 - learning_rate: 0.0010
Epoch 11/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.4914 - loss: 1.8567
Epoch 11: val_accuracy improved from 0.50000 to 0.50820, saving model to modelo_v3.keras

Epoch 11: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.4932 - loss: 1.8469 - val_accuracy: 0.5082 - val_loss: 1.8119 - learning_rate: 0.0010
Epoch 12/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5049 - loss: 1.7871
Epoch 12: val_accuracy did not improve from 0.50820
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.5066 - loss: 1.7865 - val_accuracy: 0.4774 - val_loss: 2.0079 - learning_rate: 0.0010
Epoch 13/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5195 - loss: 1.7356
Epoch 13: val_accuracy improved from 0.50820 to 0.50900, saving model to modelo_v3.keras

Epoch 13: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.5195 - loss: 1.7412 - val_accuracy: 0.5090 - val_loss: 1.7955 - learning_rate: 0.0010
Epoch 14/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5324 - loss: 1.6782
Epoch 14: val_accuracy improved from 0.50900 to 0.51120, saving model to modelo_v3.keras

Epoch 14: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.5300 - loss: 1.6881 - val_accuracy: 0.5112 - val_loss: 1.8342 - learning_rate: 0.0010
Epoch 15/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.5408 - loss: 1.6402
Epoch 15: val_accuracy improved from 0.51120 to 0.53500, saving model to modelo_v3.keras

Epoch 15: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.5400 - loss: 1.6485 - val_accuracy: 0.5350 - val_loss: 1.7171 - learning_rate: 0.0010
Epoch 16/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5507 - loss: 1.6045
Epoch 16: val_accuracy improved from 0.53500 to 0.53780, saving model to modelo_v3.keras

Epoch 16: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.5496 - loss: 1.6119 - val_accuracy: 0.5378 - val_loss: 1.7661 - learning_rate: 0.0010
Epoch 17/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5625 - loss: 1.5594
Epoch 17: val_accuracy improved from 0.53780 to 0.55620, saving model to modelo_v3.keras

Epoch 17: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.5586 - loss: 1.5747 - val_accuracy: 0.5562 - val_loss: 1.6107 - learning_rate: 0.0010
Epoch 18/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5679 - loss: 1.5253
Epoch 18: val_accuracy did not improve from 0.55620
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.5674 - loss: 1.5362 - val_accuracy: 0.5332 - val_loss: 1.8866 - learning_rate: 0.0010
Epoch 19/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5780 - loss: 1.4915
Epoch 19: val_accuracy improved from 0.55620 to 0.57440, saving model to modelo_v3.keras

Epoch 19: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.5750 - loss: 1.5057 - val_accuracy: 0.5744 - val_loss: 1.5696 - learning_rate: 0.0010
Epoch 20/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5879 - loss: 1.4550
Epoch 20: val_accuracy did not improve from 0.57440
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.5808 - loss: 1.4763 - val_accuracy: 0.5722 - val_loss: 1.5899 - learning_rate: 0.0010
Epoch 21/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5884 - loss: 1.4516
Epoch 21: val_accuracy did not improve from 0.57440
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.5868 - loss: 1.4580 - val_accuracy: 0.5620 - val_loss: 1.6483 - learning_rate: 0.0010
Epoch 22/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.5964 - loss: 1.4171
Epoch 22: val_accuracy did not improve from 0.57440
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.5943 - loss: 1.4305 - val_accuracy: 0.5226 - val_loss: 1.8255 - learning_rate: 0.0010
Epoch 23/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6031 - loss: 1.3944
Epoch 23: val_accuracy improved from 0.57440 to 0.59300, saving model to modelo_v3.keras

Epoch 23: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6010 - loss: 1.4045 - val_accuracy: 0.5930 - val_loss: 1.5034 - learning_rate: 0.0010
Epoch 24/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6082 - loss: 1.3738
Epoch 24: val_accuracy improved from 0.59300 to 0.61680, saving model to modelo_v3.keras

Epoch 24: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6042 - loss: 1.3816 - val_accuracy: 0.6168 - val_loss: 1.4005 - learning_rate: 0.0010
Epoch 25/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6170 - loss: 1.3448
Epoch 25: val_accuracy improved from 0.61680 to 0.61740, saving model to modelo_v3.keras

Epoch 25: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6128 - loss: 1.3521 - val_accuracy: 0.6174 - val_loss: 1.3858 - learning_rate: 0.0010
Epoch 26/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6257 - loss: 1.3172
Epoch 26: val_accuracy did not improve from 0.61740
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6210 - loss: 1.3369 - val_accuracy: 0.5694 - val_loss: 1.6276 - learning_rate: 0.0010
Epoch 27/150
1404/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6273 - loss: 1.2847
Epoch 27: val_accuracy did not improve from 0.61740
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6214 - loss: 1.3168 - val_accuracy: 0.6038 - val_loss: 1.4853 - learning_rate: 0.0010
Epoch 28/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6305 - loss: 1.2842
Epoch 28: val_accuracy did not improve from 0.61740
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6278 - loss: 1.2949 - val_accuracy: 0.6026 - val_loss: 1.4699 - learning_rate: 0.0010
Epoch 29/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6339 - loss: 1.2756
Epoch 29: val_accuracy did not improve from 0.61740
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6342 - loss: 1.2753 - val_accuracy: 0.6030 - val_loss: 1.4831 - learning_rate: 0.0010
Epoch 30/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6428 - loss: 1.2384
Epoch 30: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 30: val_accuracy did not improve from 0.61740
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6411 - loss: 1.2492 - val_accuracy: 0.6104 - val_loss: 1.4763 - learning_rate: 0.0010
Epoch 31/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6642 - loss: 1.1641
Epoch 31: val_accuracy improved from 0.61740 to 0.62020, saving model to modelo_v3.keras

Epoch 31: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6644 - loss: 1.1515 - val_accuracy: 0.6202 - val_loss: 1.3999 - learning_rate: 5.0000e-04
Epoch 32/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6761 - loss: 1.1050
Epoch 32: val_accuracy improved from 0.62020 to 0.63380, saving model to modelo_v3.keras

Epoch 32: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6721 - loss: 1.1180 - val_accuracy: 0.6338 - val_loss: 1.3344 - learning_rate: 5.0000e-04
Epoch 33/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6809 - loss: 1.0857
Epoch 33: val_accuracy improved from 0.63380 to 0.64740, saving model to modelo_v3.keras

Epoch 33: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6803 - loss: 1.0880 - val_accuracy: 0.6474 - val_loss: 1.3071 - learning_rate: 5.0000e-04
Epoch 34/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6892 - loss: 1.0574
Epoch 34: val_accuracy did not improve from 0.64740
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6856 - loss: 1.0703 - val_accuracy: 0.6190 - val_loss: 1.4254 - learning_rate: 5.0000e-04
Epoch 35/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6900 - loss: 1.0460
Epoch 35: val_accuracy improved from 0.64740 to 0.65460, saving model to modelo_v3.keras

Epoch 35: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6869 - loss: 1.0598 - val_accuracy: 0.6546 - val_loss: 1.3159 - learning_rate: 5.0000e-04
Epoch 36/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6932 - loss: 1.0360
Epoch 36: val_accuracy did not improve from 0.65460
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6926 - loss: 1.0429 - val_accuracy: 0.6270 - val_loss: 1.4409 - learning_rate: 5.0000e-04
Epoch 37/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7012 - loss: 1.0165
Epoch 37: val_accuracy did not improve from 0.65460
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6981 - loss: 1.0307 - val_accuracy: 0.6434 - val_loss: 1.3250 - learning_rate: 5.0000e-04
Epoch 38/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6986 - loss: 1.0121
Epoch 38: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 38: val_accuracy did not improve from 0.65460
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.6963 - loss: 1.0186 - val_accuracy: 0.6426 - val_loss: 1.3565 - learning_rate: 5.0000e-04
Epoch 39/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7077 - loss: 0.9778
Epoch 39: val_accuracy improved from 0.65460 to 0.66360, saving model to modelo_v3.keras

Epoch 39: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7100 - loss: 0.9687 - val_accuracy: 0.6636 - val_loss: 1.2714 - learning_rate: 2.5000e-04
Epoch 40/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7185 - loss: 0.9439
Epoch 40: val_accuracy did not improve from 0.66360
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7181 - loss: 0.9509 - val_accuracy: 0.6564 - val_loss: 1.2859 - learning_rate: 2.5000e-04
Epoch 41/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7247 - loss: 0.9389
Epoch 41: val_accuracy did not improve from 0.66360
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7220 - loss: 0.9440 - val_accuracy: 0.6578 - val_loss: 1.3157 - learning_rate: 2.5000e-04
Epoch 42/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7285 - loss: 0.9090
Epoch 42: val_accuracy did not improve from 0.66360
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7250 - loss: 0.9223 - val_accuracy: 0.6618 - val_loss: 1.2841 - learning_rate: 2.5000e-04
Epoch 43/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7235 - loss: 0.9175
Epoch 43: val_accuracy did not improve from 0.66360
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7234 - loss: 0.9181 - val_accuracy: 0.6552 - val_loss: 1.3255 - learning_rate: 2.5000e-04
Epoch 44/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7291 - loss: 0.9084
Epoch 44: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.

Epoch 44: val_accuracy did not improve from 0.66360
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7318 - loss: 0.8978 - val_accuracy: 0.6624 - val_loss: 1.3150 - learning_rate: 2.5000e-04
Epoch 45/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7358 - loss: 0.8804
Epoch 45: val_accuracy did not improve from 0.66360
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7360 - loss: 0.8767 - val_accuracy: 0.6604 - val_loss: 1.3073 - learning_rate: 1.2500e-04
Epoch 46/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7418 - loss: 0.8652
Epoch 46: val_accuracy did not improve from 0.66360
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7392 - loss: 0.8692 - val_accuracy: 0.6620 - val_loss: 1.2872 - learning_rate: 1.2500e-04
Epoch 47/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7406 - loss: 0.8622
Epoch 47: val_accuracy improved from 0.66360 to 0.66880, saving model to modelo_v3.keras

Epoch 47: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7407 - loss: 0.8650 - val_accuracy: 0.6688 - val_loss: 1.2619 - learning_rate: 1.2500e-04
Epoch 48/150
1404/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7423 - loss: 0.8554
Epoch 48: val_accuracy did not improve from 0.66880
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7415 - loss: 0.8618 - val_accuracy: 0.6656 - val_loss: 1.2835 - learning_rate: 1.2500e-04
Epoch 49/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7417 - loss: 0.8523
Epoch 49: val_accuracy did not improve from 0.66880
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7442 - loss: 0.8487 - val_accuracy: 0.6628 - val_loss: 1.3135 - learning_rate: 1.2500e-04
Epoch 50/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7424 - loss: 0.8588
Epoch 50: val_accuracy did not improve from 0.66880
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7422 - loss: 0.8540 - val_accuracy: 0.6596 - val_loss: 1.3061 - learning_rate: 1.2500e-04
Epoch 51/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7466 - loss: 0.8378
Epoch 51: val_accuracy did not improve from 0.66880
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7450 - loss: 0.8419 - val_accuracy: 0.6626 - val_loss: 1.2989 - learning_rate: 1.2500e-04
Epoch 52/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7473 - loss: 0.8384
Epoch 52: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.

Epoch 52: val_accuracy did not improve from 0.66880
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7455 - loss: 0.8426 - val_accuracy: 0.6662 - val_loss: 1.2657 - learning_rate: 1.2500e-04
Epoch 53/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7475 - loss: 0.8227
Epoch 53: val_accuracy did not improve from 0.66880
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.7483 - loss: 0.8266 - val_accuracy: 0.6680 - val_loss: 1.2682 - learning_rate: 6.2500e-05
Epoch 54/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7533 - loss: 0.8172
Epoch 54: val_accuracy did not improve from 0.66880
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.7518 - loss: 0.8159 - val_accuracy: 0.6636 - val_loss: 1.2901 - learning_rate: 6.2500e-05
Epoch 55/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7536 - loss: 0.8144
Epoch 55: val_accuracy did not improve from 0.66880
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 18ms/step - accuracy: 0.7523 - loss: 0.8185 - val_accuracy: 0.6648 - val_loss: 1.2826 - learning_rate: 6.2500e-05
Epoch 56/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7501 - loss: 0.8136
Epoch 56: val_accuracy did not improve from 0.66880
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 26s 19ms/step - accuracy: 0.7511 - loss: 0.8161 - val_accuracy: 0.6652 - val_loss: 1.2813 - learning_rate: 6.2500e-05
Epoch 57/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7527 - loss: 0.8102
Epoch 57: val_accuracy improved from 0.66880 to 0.67600, saving model to modelo_v3.keras

Epoch 57: finished saving model to modelo_v3.keras
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7541 - loss: 0.8093 - val_accuracy: 0.6760 - val_loss: 1.2523 - learning_rate: 6.2500e-05
Epoch 58/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7511 - loss: 0.8211
Epoch 58: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7533 - loss: 0.8120 - val_accuracy: 0.6656 - val_loss: 1.2901 - learning_rate: 6.2500e-05
Epoch 59/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7581 - loss: 0.7953
Epoch 59: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7555 - loss: 0.8022 - val_accuracy: 0.6668 - val_loss: 1.2813 - learning_rate: 6.2500e-05
Epoch 60/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7548 - loss: 0.7980
Epoch 60: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7540 - loss: 0.8017 - val_accuracy: 0.6676 - val_loss: 1.2872 - learning_rate: 6.2500e-05
Epoch 61/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7586 - loss: 0.7961
Epoch 61: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7576 - loss: 0.7995 - val_accuracy: 0.6714 - val_loss: 1.2777 - learning_rate: 6.2500e-05
Epoch 62/150
1404/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7601 - loss: 0.7930
Epoch 62: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.

Epoch 62: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7583 - loss: 0.7992 - val_accuracy: 0.6636 - val_loss: 1.2989 - learning_rate: 6.2500e-05
Epoch 63/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7561 - loss: 0.7998
Epoch 63: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7582 - loss: 0.7968 - val_accuracy: 0.6706 - val_loss: 1.2742 - learning_rate: 3.1250e-05
Epoch 64/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7651 - loss: 0.7727
Epoch 64: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7618 - loss: 0.7871 - val_accuracy: 0.6740 - val_loss: 1.2648 - learning_rate: 3.1250e-05
Epoch 65/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7609 - loss: 0.7876
Epoch 65: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7595 - loss: 0.7901 - val_accuracy: 0.6730 - val_loss: 1.2647 - learning_rate: 3.1250e-05
Epoch 66/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7596 - loss: 0.7916
Epoch 66: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7604 - loss: 0.7917 - val_accuracy: 0.6710 - val_loss: 1.2790 - learning_rate: 3.1250e-05
Epoch 67/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7586 - loss: 0.7847
Epoch 67: ReduceLROnPlateau reducing learning rate to 1.5625000742147677e-05.

Epoch 67: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7595 - loss: 0.7886 - val_accuracy: 0.6696 - val_loss: 1.2805 - learning_rate: 3.1250e-05
Epoch 68/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7619 - loss: 0.7821
Epoch 68: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7620 - loss: 0.7862 - val_accuracy: 0.6698 - val_loss: 1.2844 - learning_rate: 1.5625e-05
Epoch 69/150
1405/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7609 - loss: 0.7917
Epoch 69: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7612 - loss: 0.7832 - val_accuracy: 0.6706 - val_loss: 1.2876 - learning_rate: 1.5625e-05
Epoch 70/150
1406/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7604 - loss: 0.7787
Epoch 70: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7624 - loss: 0.7776 - val_accuracy: 0.6730 - val_loss: 1.2738 - learning_rate: 1.5625e-05
Epoch 71/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7658 - loss: 0.7823
Epoch 71: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7632 - loss: 0.7803 - val_accuracy: 0.6730 - val_loss: 1.2785 - learning_rate: 1.5625e-05
Epoch 72/150
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7634 - loss: 0.7833
Epoch 72: ReduceLROnPlateau reducing learning rate to 7.812500371073838e-06.

Epoch 72: val_accuracy did not improve from 0.67600
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 25s 18ms/step - accuracy: 0.7645 - loss: 0.7797 - val_accuracy: 0.6710 - val_loss: 1.2813 - learning_rate: 1.5625e-05
Epoch 72: early stopping
Restoring model weights from the end of the best epoch: 57.
